# Tennis Vision — pipeline benchmark on a Colab T4

Measures the reusable pipeline (`pipeline.run.run_analysis`) on a GPU:

- frames processed per second (detection, and the detectors alone);
- time per stage and end-to-end;
- GPU memory (PyTorch allocator peak and `nvidia-smi` peak);
- peak system RAM;
- an **estimate** for a 10-minute video from the measured per-frame costs.

**Before running:** set *Runtime → Change runtime type → T4 GPU*. Put a video (fixed, elevated, behind-baseline camera; ≤ 10 min; MP4/MOV) at `MyDrive/tennis_vision/input_videos/`. Weights and work directories are cached on Drive, so a disconnected session resumes.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No CUDA device: switch the runtime to a GPU."
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Drive, code, dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib, shutil, subprocess, json
DRIVE = pathlib.Path('/content/drive/MyDrive/tennis_vision')
for sub in ('input_videos', 'models', 'benchmarks', 'work'):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

REPO_URL = 'https://github.com/Curious-Mahdi/Tennis_cv.git'
BRANCH = 'main'                      # the branch that contains pipeline/
REPO = pathlib.Path('/content/Tennis_cv')
if not REPO.exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip -q install -e .
assert (REPO / 'pipeline' / 'run.py').exists(), 'This branch has no pipeline/ package yet.'

## 3. Model weights (cached on Drive)

In [ ]:
WEIGHTS = ['models/keypoints_model_geoaug.pth', 'models/pose_landmarker_lite.task', 'models/tracknet.pt', 'yolov8x.pt']
for w in WEIGHTS:
    cached = DRIVE / 'models' / pathlib.Path(w).name
    if cached.exists() and not (REPO / w).exists():
        shutil.copy(cached, REPO / w)
!python scripts/download_models.py || true
if not (REPO / 'models/tracknet.pt').exists():
    !gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O models/tracknet.pt   # TrackNet, from its author
from ultralytics import YOLO; YOLO(str(REPO / 'yolov8x.pt'))          # downloads once if missing
for w in WEIGHTS:
    if (REPO / w).exists():
        shutil.copy(REPO / w, DRIVE / 'models' / pathlib.Path(w).name)
!ls -la models yolov8x.pt

## 4. Choose the video

In [ ]:
videos = sorted(p for p in (DRIVE / 'input_videos').iterdir() if p.suffix.lower() in ('.mp4', '.mov', '.m4v'))
assert videos, 'Upload a video to MyDrive/tennis_vision/input_videos/'
VIDEO = videos[0]
from pipeline.video import probe, plan
info = probe(VIDEO); print(info); print(plan(info))     # fails here, with a reason, if unsupported

## 5. Benchmark

Each run starts from a clean work directory on local disk (Drive I/O would distort the timing). 300 frames is about 10 s of play; the second run checks that the rate is stable.

In [ ]:
results = {}
for frames in (300, 900):
    work = pathlib.Path(f'/content/bench_{frames}/work')
    !python scripts/benchmark_pipeline.py --video "{VIDEO}" --frames {frames} --workdir {work}
    results[frames] = json.loads((work.parent / 'benchmark.json').read_text())
    shutil.copy(work.parent / 'benchmark.json', DRIVE / 'benchmarks' / f'benchmark_t4_{frames}.json')

In [ ]:
import pandas as pd
rows = []
for frames, r in results.items():
    m, e = r['measured'], r['estimate_10_min_video']
    rows.append({'analysed frames': frames, 'detection fps': m['detection_fps'], 'detectors-only fps': m['detector_only_fps'],
                 'end-to-end s': m['end_to_end_s'], 'peak RAM MB': m['peak_ram_mb'],
                 'GPU peak allocated MB': (m['gpu_memory_torch'] or {}).get('peak_allocated_mb'),
                 'GPU nvidia-smi peak MB': m['gpu_memory_nvidia_smi_peak_mb'],
                 'EST. 10-min video (min)': e['total_minutes']})
pd.DataFrame(rows)

## 6. Optional: full analysis of the video

Runs every analysable segment, with the work directory on Drive so it resumes after a disconnect (re-run the cell). Outputs: `results/analysis.json`, per-segment JSON/CSV, `results/annotated.mp4`.

In [ ]:
from pipeline.models import ModelBundle
from pipeline.run import run_analysis
from tqdm.auto import tqdm
models = ModelBundle()
work = DRIVE / 'work' / VIDEO.stem
bar = tqdm(total=100, unit='%')
def progress(e):
    bar.n = round(100 * e['progress']); bar.set_description(e['stage']); bar.refresh()
index = run_analysis(VIDEO, work, on_progress=progress, models=models)
bar.close()
print(json.dumps(index['segments'], indent=2))